# Lab 1: Models and Deployment (15 min)

## Objectives
- Understand the Foundry model catalog
- Use the `azure-ai-inference` SDK for chat calls
- Explore parameters: temperature, max_tokens, system prompt
- Compare model behaviors

## Concepts

### Model Catalog
Foundry provides 1600+ models from different providers:
- **OpenAI**: GPT-4o, GPT-4o-mini, o1, o3
- **Meta**: Llama 3.1, 3.2, 3.3
- **Mistral**: Mistral Large, Small
- **Microsoft**: Phi-4, MAI

### Deployment Types
| Type | Description | When to use |
|------|-------------|-------------|
| **Standard** | Pay-as-you-go, shared | Development, low volume |
| **Provisioned** | Reserved throughput | Production, predictable latency |
| **Serverless** | MaaS models (pay-per-token) | Third-party models |

### Unified SDK
In Foundry, we use `azure-ai-inference` for all model calls:
```python
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential

client = ChatCompletionsClient(
    endpoint=FOUNDRY_ENDPOINT,
    credential=AzureKeyCredential(FOUNDRY_KEY),
)
```

In [ ]:
# Setup - load configuration
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

endpoint_base = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT", "").rstrip("/")
key = os.getenv("AZURE_AI_FOUNDRY_KEY", "")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

# For cognitiveservices.azure.com or openai.azure.com endpoints,
# the deployment must be embedded in the endpoint (don't pass model= in .complete())
if "cognitiveservices.azure.com" in endpoint_base or "openai.azure.com" in endpoint_base:
    endpoint = f"{endpoint_base}/openai/deployments/{model}"
else:
    endpoint = endpoint_base if endpoint_base.endswith("/models") else endpoint_base + "/models"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
)

print(f"✅ Client created for: {endpoint}")
print(f"   Model: {model}")

✅ Cliente criado para: https://foundry-mod2.cognitiveservices.azure.com/openai/deployments/gpt-4o
   Modelo: gpt-4o


## 🖥️ Exercise 1.1: First Model Call

Let's make a simple call with a system message (system prompt).

In [ ]:
# Exercise 1.1: Call with system prompt
response = client.complete(
    messages=[
        SystemMessage(content="You are an Azure specialist assistant. Always respond in English."),
        UserMessage(content="What is Azure AI Foundry? Explain in 3 sentences."),
    ],
    max_tokens=200,
    temperature=0.7,
)

print("=== Response ===")
print(response.choices[0].message.content)
print(f"\n📊 Tokens: prompt={response.usage.prompt_tokens}, response={response.usage.completion_tokens}, total={response.usage.total_tokens}")

=== Resposta ===
O Azure AI Foundry é uma iniciativa da Microsoft que visa acelerar a inovação em inteligência artificial (IA) através de colaboração com empresas e parceiros estratégicos. Focado em criar soluções de IA personalizadas e escaláveis, combina tecnologias avançadas do Azure com expertise técnica e de negócio. Este programa permite que organizações desenvolvam e implementem rapidamente soluções impulsionadas por IA para resolver desafios únicos e gerar impacto real.

📊 Tokens: prompt=43, resposta=84, total=127


## 🖥️ Exercise 1.2: Streaming

In real applications, we use streaming to display text as it's being generated.

In [ ]:
# Exercise 1.2: Streaming response
print("=== Streaming ===")

response = client.complete(
    messages=[
        SystemMessage(content="You are a creative poet."),
        UserMessage(content="Write a haiku about cloud computing."),
    ],
    max_tokens=100,
    stream=True,
)

for chunk in response:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

print("\n\n✅ Streaming complete!")

=== Streaming ===
Nuvens digitais,  
dados dançam no éter,  
mundo conectado.

✅ Streaming concluído!


## 🖥️ Exercise 1.3: Multi-turn Conversation

Models maintain context when you send the conversation history.

In [ ]:
# Exercise 1.3: Conversation with history
from azure.ai.inference.models import AssistantMessage

history = [
    SystemMessage(content="You are an Azure technical assistant. Respond concisely."),
    UserMessage(content="What is a Resource Group?"),
]

# First question
r1 = client.complete(messages=history, max_tokens=150)
answer1 = r1.choices[0].message.content
print("👤 What is a Resource Group?")
print(f"🤖 {answer1}\n")

# Add to history and follow up
history.append(AssistantMessage(content=answer1))
history.append(UserMessage(content="And how does it relate to a subscription?"))

r2 = client.complete(messages=history, max_tokens=150)
print("👤 And how does it relate to a subscription?")
print(f"🤖 {r2.choices[0].message.content}")

👤 O que é um Resource Group?
🤖 Um **Resource Group** no Azure é um contêiner lógico que agrupa recursos relacionados, como máquinas virtuais, bancos de dados e redes. Ele serve para organizar, gerenciar e aplicar configurações ou permissões a esses recursos de forma conjunta. Além disso, todos os recursos em um Resource Group compartilham o mesmo ciclo de vida, facilitando o gerenciamento e a exclusão em massa.

👤 E como se relaciona com uma subscription?
🤖 Um **Resource Group** está sempre associado a uma **Subscription** no Azure. A **Subscription** é o nível superior de organização que define os limites de faturamento e acesso ao Azure. Dentro de uma subscription, você pode criar múltiplos Resource Groups, que, por sua vez, contêm os recursos individuais. Em resumo: os **recursos estão dentro de Resource Groups, e os Resource Groups estão dentro de uma Subscription**.


## 🖥️ Exercise 1.4: Comparing Temperature

**Temperature** controls the randomness of responses:
- `0.0` → Deterministic, factual
- `1.0` → Creative, varied

In [ ]:
# Exercise 1.4: Compare temperatures
prompt = "Suggest a name for an AI healthcare startup."

for temp in [0.0, 0.5, 1.0]:
    response = client.complete(
        messages=[UserMessage(content=prompt)],
        max_tokens=50,
        temperature=temp,
    )
    print(f"🌡️ Temperature {temp}: {response.choices[0].message.content.strip()}")
    print()

🌡️ Temperature 0.0: Sure! Que tal **"MedIntel"**?  

O nome combina "Med" (de medicina) com "Intel" (de inteligência, remetendo à inteligência artificial), transmitindo inovação, tecnologia e foco na saúde.

🌡️ Temperature 0.5: **NeuralCare**  

Esse nome combina a ideia de inteligência artificial (neural, remetendo a redes neurais) com cuidado e saúde (care), transmitindo inovação e um foco humano.

🌡️ Temperature 1.0: Sure! Que tal **"MedIntelix"**?  

Esse nome combina "Medicine" (medicina) com "Intelligence" e "X" para representar tecnologia de ponta e inovação, transmitindo modernidade e foco em inteligência artificial aplicada



## 🧪 Extra Challenge

Try different system prompts to see how they change the responses:
- `"You are a pirate who only speaks in rhymes"`
- `"Always respond with exactly 3 bullet points"`
- `"You are a senior engineer reviewing code"`

In [ ]:
# Challenge: try it here!
response = client.complete(
    model=model,
    messages=[
        SystemMessage(content="Always respond with exactly 3 bullet points"),
        UserMessage(content="What are the advantages of using Azure AI Foundry?"),
    ],
    max_tokens=200,
)

print(response.choices[0].message.content)

## ✅ Summary

In this lab you learned:
- How to use the `ChatCompletionsClient` from `azure-ai-inference`
- Simple calls, streaming, and multi-turn conversations
- The impact of temperature and system prompts
- That the SDK is the same regardless of the model

**Next:** [Lab 2 - Agents](../lab02/lab02-agentes.ipynb)